# GAWASCO Survey Data Pipeline — Combined (All-in-One)

This single notebook merges the two GAWASCO notebooks end-to-end, in the order the work is actually meant to happen:

1. **Build the clean database** — append this run's raw export, classify every no-account-number row as **Potential Customer** or **Reject**, remove duplicate account numbers, match against the company account database, save the clean workbook, then **correct it for wrong-GPS submissions** (records whose GPS point landed on the road instead of the customer's premise get moved from `All Data (Clean)` into `Rejects`), and finally add a **`Zone Progress` sheet** — a live coverage tracker by zone (total/visited/not-visited/coverage %, colour-scaled, with a bar chart) — right into the workbook. This step produces the one "true" clean database for the run.
2. **Progress report** — a Word summary of coverage, this run's processing stats, zone-wise coverage (table + a colour-coded coverage-by-zone chart), and daily enumerator performance, built from the *corrected* clean database.
3. **Remaining accounts** — a PDF of accounts not yet visited (zone → route), also built from the *corrected* clean database.
4. **Weekly submissions report** — a supporting Excel breakdown of accepted/rejected submissions per collector, per day, for a chosen date range.
5. **Day-specific extras** — an Enumerator Submission Summary sheet, a Reject List sheet, and a narrative Word analysis report, all filtered to one specific date (`REPORT_DATE`) and added onto the same corrected workbook.

**Why this order matters:** the not-visited PDF and the progress report now read the database *after* GPS-correction, so an account whose only submission got GPS-rejected correctly shows up as "not yet visited" / a rejection, rather than being counted as clean before the correction ever ran.

**One consequence of reordering:** since Steps 2–5 all read the clean database back from disk *after* it's been saved and GPS-corrected, "today's" accepted/rejected counts (used in the progress report and the Enumerator Summary sheet) are computed by filtering the saved workbook on `REPORT_DATE`, rather than an in-memory tag. Both places that need "today's" counts now share one function (`build_daily_counts`), so the numbers can't drift apart.

**Dates:** every date column (**submission time** and **interview date**) is displayed as `dd/mm/yyyy` throughout the output workbooks. For compiling a day's enumerator output — the daily counts, the Reject List sheet, and the Word analysis report — **submission time is the primary date**; interview date is only used as a fallback if submission time is blank. Set `REPORT_DATE` to the actual date `RAW_FILE` was submitted on so "today's" counts line up.

**How to use this each run:**
1. Update the **Configuration** cell (file paths, dates, `REPORT_DATE`, `REPORT_DATE_LABEL(_FULL)`).
2. Run All. If you don't have a wrong-GPS file for this run, set `INCLUDE_GPS_CORRECTION_STAGE = False` — the pipeline still runs end-to-end, just skips the correction.

**Potential vs Reject rule:** a row with no account number is a **Potential Customer** if `Q1. Is GAWASCO Service available at this property?` is `No`. Any other no-account row is a **Reject**. This rule is re-applied to the *entire* dataset each run.

**Handling form changes (Q-number drift):** raw columns are reconciled to the master schema **by question text**, ignoring the `Q<number>.` prefix, so form renumbering doesn't silently lose or scramble data.


## 0. Setup

In [1]:
import collections
import os
import re
from datetime import datetime

import numpy as np
import pandas as pd

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import BarChart, Reference

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from docx import Document
from docx.shared import Pt, Cm, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)


## 1. Configuration

Update these every run. Everything else in the notebook adapts automatically.

**Files**
- `RAW_FILE`: this run's manually-cleaned raw survey export (single sheet).
- `CLEAN_FILE`: the running master workbook from **last run**, with `All Data (Clean)` and `Rejects` sheets. This run's corrected clean workbook becomes next run's `CLEAN_FILE`. (If `CLEAN_FILE` still has the old `Duplicate Account Numbers` sheet, it's loaded automatically and renamed to `Rejects` going forward.)
- `DB_FILE`: the latest export from the company's account database (`Result 1` sheet).
- `OUTPUT_DIR`: where all output files are written.

**GPS correction**
- `INCLUDE_GPS_CORRECTION_STAGE`: set `False` to skip GPS correction for a run that has no wrong-GPS file — everything else still runs, just against the un-corrected clean database.
- `WRONG_GPS_WORKBOOK`: the "Wrong Coordinates" workbook flagging records whose GPS point landed on the road rather than the premise.
- `GPS_STAGE_OUTPUT`: where the corrected workbook is saved (defaults to overwriting the just-written clean workbook).

**Day-specific reporting date** (drives the progress report's daily table, the Enumerator Summary sheet, the Reject List sheet, and the Word analysis report)
- `REPORT_DATE`: the date (ISO `YYYY-MM-DD`) to compile — matched against each row's **submission time** first, interview date only as a fallback — should equal the actual date `RAW_FILE` was submitted on so "today's" counts line up.
- `REPORT_DATE_LABEL`: short label used in sheet names, e.g. `"24 Aug"`.
- `REPORT_DATE_LABEL_FULL`: full label used in the Word report headline, e.g. `"24th August 2026"`.

**Weekly report date range** (used only by the Weekly Submissions Report)
- `DATE_RANGE_MODE`: `"week"` (auto Sunday–Saturday around `ANY_DATE_IN_WEEK`) or `"custom"` (`CUSTOM_START_DATE` to `CUSTOM_END_DATE`).


In [2]:
# ===========================================================================
# CONFIG — update every run
# ===========================================================================

# ----- FILES -----
RAW_FILE = "21,22 September.xlsx"                                  # this run's raw survey export
CLEAN_FILE = "GAWASCO_Clean_Combined_18th_September.xlsx"   # last run's master clean/rejects workbook
DB_FILE = "COMBINED_ACCOUNT_LISTING_ALL_ZONES_updated.xlsx"            # latest company account-database export
OUTPUT_DIR = "outputs"                                          # folder to write outputs into
RUN_DATE_LABEL = "21st September"                                  # used only in output filenames

# ----- GPS CORRECTION -----
INCLUDE_GPS_CORRECTION_STAGE = False   # set False to skip if there's no wrong-GPS file this run
WRONG_GPS_WORKBOOK = ".xlsx"  # flagged wrong-GPS records for this run
GPS_STAGE_OUTPUT = None               # None -> overwrite the clean workbook in place

# ----- DAY-SPECIFIC REPORTING DATE -----
REPORT_DATE = "2026-09-21"            # date to compile (ISO) — matched primarily on submission time, interview date as fallback
REPORT_DATE_LABEL = "21 sept"          # short label used in sheet names, e.g. "Enumerator Summary (24 Aug)"
REPORT_DATE_LABEL_FULL = "21st September 2026"  # full label used in the Word report headline

WORD_ANALYSIS_REPORT_OUT = f"{OUTPUT_DIR}/GAWASCO_{RUN_DATE_LABEL.replace(' ', '_')}_Analysis_Report.docx"

# ----- WEEKLY REPORT DATE RANGE -----
DATE_RANGE_MODE = "week"          # "week" or "custom"
ANY_DATE_IN_WEEK = "2026-09-21"   # used when DATE_RANGE_MODE == "week": picks the Sun-Sat week containing this date
CUSTOM_START_DATE = "2026-09-21"  # used only when DATE_RANGE_MODE == "custom"
CUSTOM_END_DATE = "2026-09-27"

# ----- SHEET NAMES -----
CLEAN_SHEET = "All Data (Clean)"
REJECTS_SHEET = "Rejects"
LEGACY_REJECTS_SHEET = "Duplicate Account Numbers"   # old sheet name, read automatically if REJECTS_SHEET isn't found
DB_SHEET = "Result 1"

# ----- KEY COLUMN NAMES (survey side) -----
COL_ACCOUNT = "Q3. Customer account number"
COL_CUST_NAME = "Q2. Customer full name"
COL_CUST_PHONE = "Q4. Phone number / Account Holder"
COL_ZONE = "Q5. Zone"
COL_ROUTE = "Q6. Route"
COL_RECORD_TYPE = "Record Type"
COL_SERVICE_AVAILABLE = "Q1. Is GAWASCO Service available at this property?"   # decides Potential Customer vs Reject
COL_COLLECTOR = "Q2. Data collector name"
COL_REJECT_REASON = "Reject_Reason"
DATE_COL_PRIMARY = "_submission_time"
DATE_COL_FALLBACK = "Q1. Date and time of interview"

# ----- KEY COLUMN NAMES (company DB side) -----
DB_COL_ACC = "acc"
DB_COL_NAME = "acc_name"
DB_COL_PHONE = "mobile_no"
DB_COL_ZONE = "zone"
DB_COL_ROUTE = "route"
DB_COL_STATUS = "acc_status"
DB_COL_CATEGORY = "category"

# ----- GPS coordinate columns (present on both Clean and Rejects sheets) -----
COL_GPS_LAT = "_Q7. GPS coordinates_latitude"
COL_GPS_LON = "_Q7. GPS coordinates_longitude"

# ----- Reject List columns (respondent contact fields, distinct from the
# customer-record COL_CUST_NAME / COL_CUST_PHONE used above) -----
COL_RESPONDENT_PHONE = "Q9. Respondents Phone Number"
COL_RESPONDENT_NAME = "Q8. Respondents Name "


## 2. Helper functions

- `normalize_account`: cleans an account number for comparison (trims whitespace, uppercases, treats blank/`None`/`"none"` as no account).
- `is_blank`: flags empty-looking text cells (NaN, empty string, literal "None").
- `classify_record`: applies the Account Holder / Potential Customer / Reject rule.
- `reject_reason_for`: a short, human-readable reason recorded on the `Rejects` sheet for no-account rows.
- `parse_any_date` / `sunday_saturday_week`: handle mixed date formats and work out Sunday–Saturday week boundaries.
- `normalize_colname` / `reconcile_columns`: match raw columns to the master schema **by question text**, ignoring `Q<number>.` prefixes.
- `round_coord`: rounds a GPS coordinate to 6dp for reliable matching, or `None` if it isn't numeric.
- `coerce_dates`: parses the submission-time and interview-date columns into real (timezone-naive) datetimes before writing to Excel, so a `dd/mm/yyyy` display format actually takes effect.
- `apply_date_number_format`: applies the `dd/mm/yyyy` display format to those columns' cells in a saved workbook.
- `matches_report_date`: the single place that decides whether a row belongs to `REPORT_DATE` — **submission time is the primary date; interview date is only used as a fallback** if submission time is blank.
- `build_daily_counts`: reads a saved clean/rejects workbook and returns each collector's Accepted/Rejected counts for `REPORT_DATE` (via `matches_report_date`) — the single source of truth used by both the progress report and the Enumerator Summary sheet.


In [3]:
# ----- account / classification helpers -----

def normalize_account(x):
    'Return a comparable account-number key, or None if the cell is blank.'
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "none":
        return None
    return s


def is_blank(series):
    'Boolean mask for empty-looking text cells.'
    s = series.astype(str)
    return series.isna() | (s.str.strip() == "") | (s.str.lower() == "none")


def classify_record(acc_key, service_available):
    'Account Holder / Potential Customer / Reject, per the rule in the intro cell.'
    if pd.notna(acc_key):
        return "Account Holder"
    sa = str(service_available).strip().lower()
    if sa == "no":
        return "Potential Customer"
    return "Reject"


def reject_reason_for(service_available):
    sa = str(service_available).strip().lower()
    if sa == "yes":
        return "No account number captured (service marked available)"
    return "No account number captured (service availability not recorded)"


# ----- date helpers -----

def parse_any_date(x):
    """Robustly parse a date cell of unknown/mixed type (real datetime, text, or Excel serial number)."""
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime)):
        return pd.Timestamp(x)
    if isinstance(x, (int, float)):
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(x, unit="D")
    return pd.to_datetime(x, errors="coerce", dayfirst=False)


def sunday_saturday_week(any_date):
    """Return (Sunday, Saturday) Timestamps for the week containing any_date."""
    d = pd.Timestamp(any_date).normalize()
    offset = (d.weekday() + 1) % 7   # pandas weekday: Mon=0..Sun=6; days since the most recent Sunday
    start = d - pd.Timedelta(days=offset)
    end = start + pd.Timedelta(days=6)
    return start, end


# ----- column reconciliation (Q-number drift) -----

def normalize_colname(c):
    """Strip a leading 'Q<number><optional letter>.' prefix (and leading underscore) so columns
    can be matched by question text even when the form's Q-numbering has changed."""
    s = str(c)
    s = re.sub(r"^_?Q\d+[a-zA-Z]?\.?\s*", "", s)
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def reconcile_columns(master_cols, raw_df, verbose=True):
    """Rename raw_df's columns to match master_cols wherever the question text matches
    (ignoring Q-number prefixes), so a renumbered form still aligns correctly. Where several
    raw columns normalize to the same text (e.g. a repeat-group duplicate artifact), the
    leftmost/first one is treated as the real field. Columns that don't match anything in
    master_cols are left as-is and reported as new; they get appended to the schema rather
    than dropped, so nothing from the raw file is silently lost.
    Returns (raw_df_renamed, rename_map, new_columns, missing_from_raw).
    """
    master_norm_to_col = {}
    for c in master_cols:
        n = normalize_colname(c)
        master_norm_to_col.setdefault(n, c)   # first (leftmost) master column wins on collision

    raw_groups = {}
    for c in raw_df.columns:
        raw_groups.setdefault(normalize_colname(c), []).append(c)

    rename_map = {}
    for n, cols in raw_groups.items():
        if n in master_norm_to_col:
            rename_map[cols[0]] = master_norm_to_col[n]   # leftmost raw occurrence -> matched master column

    raw_renamed = raw_df.rename(columns=rename_map)

    # a repeat-group duplicate artifact can literally share its raw name with the rename target
    # (e.g. a stray 'Q6. Route' column colliding with 'Q5. Route' renamed to 'Q6. Route'); make unique
    col_series = pd.Series(raw_renamed.columns)
    for dup_name in col_series[col_series.duplicated()].unique():
        dup_positions = col_series[col_series == dup_name].index.values.tolist()
        for i, pos in enumerate(dup_positions):
            if i == 0:
                continue
            col_series[pos] = f"{dup_name} (extra copy {i+1})"
    raw_renamed.columns = col_series

    new_columns = [c for c in raw_renamed.columns if c not in master_cols]
    matched_master_cols = set(rename_map.values())
    missing_from_raw = [c for c in master_cols if c not in matched_master_cols]

    if verbose:
        renamed_only = {k: v for k, v in rename_map.items() if k != v}
        if renamed_only:
            print(f"  Columns matched by text despite a different name/Q-number ({len(renamed_only)}):")
            for old, new in renamed_only.items():
                print(f"    '{old}'  ->  '{new}'")
        if new_columns:
            print(f"  New raw columns not yet in the master schema, will be added ({len(new_columns)}): {new_columns}")
        if missing_from_raw:
            print(f"  Master columns this raw file doesn't have (left blank for new rows) ({len(missing_from_raw)}): {missing_from_raw}")

    return raw_renamed, rename_map, new_columns, missing_from_raw


# ----- GPS helper -----

def round_coord(v):
    'Round a GPS coordinate to 6dp for comparison, or None if it is not numeric.'
    try:
        return round(float(v), 6)
    except (TypeError, ValueError):
        return None


# ----- date formatting / "which date counts as today" helpers -----

def coerce_dates(df, cols):
    """Return a copy of df with each column in cols parsed into a real (timezone-naive)
    datetime, using parse_any_date. Excel can't store timezone-aware datetimes, so any
    timezone info is stripped (the wall-clock value is kept as-is). Doing this before writing
    to Excel is what lets a dd/mm/yyyy number format actually take effect - Excel only applies
    a date format to true date-typed cells, not to plain text."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        parsed = df[col].apply(parse_any_date)

        def _strip_tz(v):
            if pd.isna(v):
                return pd.NaT
            if isinstance(v, pd.Timestamp) and v.tzinfo is not None:
                return v.tz_localize(None)
            return v

        df[col] = parsed.apply(_strip_tz)
    return df


def apply_date_number_format(ws, headers, col_names, fmt="dd/mm/yyyy", header_row=1):
    """Apply an Excel number format to every data-row cell under the given column name(s),
    matched by header text. Safe to call again after new rows are appended (e.g. after GPS
    correction moves rows into Rejects) - only touches cells that currently hold a value."""
    for col_name in col_names:
        if col_name not in headers:
            continue
        col_idx = headers.index(col_name) + 1
        for row_num in range(header_row + 1, ws.max_row + 1):
            cell = ws.cell(row=row_num, column=col_idx)
            if cell.value is not None:
                cell.number_format = fmt


def matches_report_date(primary_val, fallback_val, report_date):
    """True if either date value falls on report_date (ISO 'YYYY-MM-DD'). Submission time
    (primary_val) is used first; interview date (fallback_val) is only consulted if the
    submission time is blank or unparseable - this is the "submission time is the primary
    date for daily enumerator output" rule."""
    d = parse_any_date(primary_val)
    if pd.isna(d):
        d = parse_any_date(fallback_val)
    if pd.isna(d):
        return False
    return d.strftime("%Y-%m-%d") == report_date


# ----- shared daily-counts helper (used by the progress report AND the Enumerator Summary sheet) -----

def build_daily_counts(workbook_path, report_date):
    """Read a saved Clean/Rejects workbook and return (accepted, rejected) Counters keyed by
    data collector, for rows whose date matches report_date - using submission time as the
    primary date and interview date only as a fallback (see matches_report_date). This is the
    single place both the progress report's daily table and the Enumerator Summary sheet pull
    from, so the two can never disagree."""
    wb = openpyxl.load_workbook(workbook_path)
    wsc = wb[CLEAN_SHEET]
    wsr = wb[REJECTS_SHEET]
    hc = [c.value for c in wsc[1]]
    hr = [c.value for c in wsr[1]]

    pi_c = hc.index(DATE_COL_PRIMARY)
    fi_c = hc.index(DATE_COL_FALLBACK)
    en_c = hc.index(COL_COLLECTOR)
    pi_r = hr.index(DATE_COL_PRIMARY)
    fi_r = hr.index(DATE_COL_FALLBACK)
    en_r = hr.index(COL_COLLECTOR)

    accepted, rejected = collections.Counter(), collections.Counter()

    for row in wsc.iter_rows(min_row=2, values_only=True):
        if matches_report_date(row[pi_c], row[fi_c], report_date):
            accepted[row[en_c]] += 1

    for row in wsr.iter_rows(min_row=2, values_only=True):
        if matches_report_date(row[pi_r], row[fi_r], report_date):
            rejected[row[en_r]] += 1

    return accepted, rejected


def daily_counts_to_frame(accepted, rejected):
    """Turn the (accepted, rejected) Counters from build_daily_counts into the same
    Enumerator x [Accepted, Rejected, Total] + TOTAL-row DataFrame shape used throughout."""
    enums = sorted(set(accepted) | set(rejected))
    frame = pd.DataFrame(
        {"Accepted": [accepted[e] for e in enums], "Rejected": [rejected[e] for e in enums]},
        index=enums,
    )
    frame["Total"] = frame["Accepted"] + frame["Rejected"]
    totals = frame.sum()
    totals.name = "TOTAL"
    return pd.concat([frame, totals.to_frame().T])


## 3. Load input files

In [4]:
raw_df = pd.read_excel(RAW_FILE, sheet_name=0)
clean_df = pd.read_excel(CLEAN_FILE, sheet_name=CLEAN_SHEET)
try:
    rejects_df = pd.read_excel(CLEAN_FILE, sheet_name=REJECTS_SHEET)
except ValueError:
    rejects_df = pd.read_excel(CLEAN_FILE, sheet_name=LEGACY_REJECTS_SHEET)
    print(f"(loaded legacy sheet '{LEGACY_REJECTS_SHEET}' — will save as '{REJECTS_SHEET}' from now on)")
db_df = pd.read_excel(DB_FILE, sheet_name=DB_SHEET)

print(f"Raw file:         {raw_df.shape[0]:>6} rows, {raw_df.shape[1]} columns")
print(f"Clean (existing): {clean_df.shape[0]:>6} rows, {clean_df.shape[1]} columns")
print(f"Rejects (existing): {rejects_df.shape[0]:>6} rows")
print(f"Company DB:       {db_df.shape[0]:>6} rows")


Raw file:            143 rows, 105 columns
Clean (existing):  14566 rows, 116 columns
Rejects (existing):   5158 rows
Company DB:        22162 rows


## 4. Build the clean database — Step 1: Append, classify, and remove duplicates

- Every raw row is joined onto the master schema, then **every row in the combined dataset (existing clean rows *and* new raw rows) is (re)classified** as `Account Holder`, `Potential Customer`, or `Reject` using the rule above — this also fixes any historical rows that were classified under the old "everything without an account number is a potential customer" logic.
- Rows classified `Reject` are removed from the clean dataset and appended to `Rejects` with a reason.
- Of what's left, rows are deduplicated **by account number only** (blank account numbers — potential customers — are never treated as duplicates of each other). For each account number that occurs more than once, the **earliest** occurrence is kept in Clean; every later occurrence is moved to `Rejects` as its own row, reason `"Duplicate account number"`.


In [5]:
raw_df = raw_df.copy()
raw_df["_source"] = "new"
clean_df = clean_df.copy()
clean_df["_source"] = "existing"

master_cols_for_matching = [c for c in clean_df.columns if c != "_source"]
print("Reconciling raw columns against the master schema (matching by question text, ignoring Q-numbers)...")
raw_matched, rename_map, extra_cols, missing_cols = reconcile_columns(master_cols_for_matching, raw_df.drop(columns="_source"))
raw_matched["_source"] = "new"

all_cols = list(dict.fromkeys(list(clean_df.columns) + list(raw_matched.columns)))
clean_df = clean_df.reindex(columns=all_cols)
raw_aligned = raw_matched.reindex(columns=all_cols)

combined = pd.concat([clean_df, raw_aligned], ignore_index=True)
combined["_acc_key"] = combined[COL_ACCOUNT].apply(normalize_account)
combined[COL_RECORD_TYPE] = [
    classify_record(k, sa) for k, sa in zip(combined["_acc_key"], combined[COL_SERVICE_AVAILABLE])
]

reject_mask = combined[COL_RECORD_TYPE] == "Reject"
reclassified_from_existing = ((combined["_source"] == "existing") & reject_mask).sum()

reject_rows = combined[reject_mask].copy()
reject_rows[COL_REJECT_REASON] = reject_rows[COL_SERVICE_AVAILABLE].apply(reject_reason_for)

keep_rows = combined[~reject_mask].copy()
dup_mask = keep_rows.duplicated(subset="_acc_key", keep="first") & keep_rows["_acc_key"].notna()
new_clean = keep_rows[~dup_mask].drop(columns="_acc_key").reset_index(drop=True)
dup_rows = keep_rows[dup_mask].drop(columns="_acc_key").copy()
dup_rows[COL_REJECT_REASON] = "Duplicate account number"

reject_rows = reject_rows.drop(columns="_acc_key")

# align the existing Rejects sheet to the current schema; a legacy (duplicates-only) sheet gets a reason filled in
rejects_df = rejects_df.copy()
if COL_REJECT_REASON not in rejects_df.columns:
    rejects_df[COL_REJECT_REASON] = "Duplicate account number"

target_cols = [c for c in new_clean.columns if c != "_source"] + [COL_REJECT_REASON]
rejects_df_aligned = rejects_df.reindex(columns=target_cols)
new_rejects_this_run = pd.concat([reject_rows, dup_rows], ignore_index=True).reindex(columns=target_cols + ["_source"])

if len(rejects_df_aligned) == 0:
    new_rejects_sheet = new_rejects_this_run.drop(columns="_source")
else:
    new_rejects_sheet = pd.concat([rejects_df_aligned, new_rejects_this_run.drop(columns="_source")], ignore_index=True)

print("STEP 1 SUMMARY")
print(f"  Raw rows this run:                              {len(raw_df):>6}")
print(f"  New columns added to the schema this run:        {len(extra_cols)}")
print(f"  Historical rows reclassified Potential -> Reject: {reclassified_from_existing:>6}")
print(f"  New duplicate account numbers found this run:    {(dup_rows['_source'] == 'new').sum():>6}")
print(f"  Clean sheet: {len(clean_df)} -> {len(new_clean)} rows")
print(f"  Rejects sheet: {len(rejects_df)} -> {len(new_rejects_sheet)} rows")

# sanity check: no account number should repeat in the clean sheet now
vc = new_clean[COL_ACCOUNT].apply(normalize_account).value_counts()
residual = (vc[vc.index.notna()] > 1).sum() if vc.index.notna().any() else 0
print(f"  Residual duplicate account numbers left in clean sheet (should be 0): {residual}")
print(f"  Potential Customer rows now in clean sheet: {(new_clean[COL_RECORD_TYPE] == 'Potential Customer').sum()}")


Reconciling raw columns against the master schema (matching by question text, ignoring Q-numbers)...
  Columns matched by text despite a different name/Q-number (37):
    'Q4. Zone'  ->  'Q5. Zone'
    'Q5. Route'  ->  'Q6. Route'
    'Q6. GPS coordinates'  ->  'Q7. GPS coordinates'
    '_Q6. GPS coordinates_latitude'  ->  '_Q7. GPS coordinates_latitude'
    '_Q6. GPS coordinates_longitude'  ->  '_Q7. GPS coordinates_longitude'
    '_Q6. GPS coordinates_altitude'  ->  '_Q7. GPS coordinates_altitude'
    '_Q6. GPS coordinates_precision'  ->  '_Q7. GPS coordinates_precision'
    'Q7. Respondents Name '  ->  'Q8. Respondents Name '
    'Q8. Respondents Phone Number'  ->  'Q9. Respondents Phone Number'
    'Q9. Is GAWASCO Service available at this property?'  ->  'Q1. Is GAWASCO Service available at this property?'
    'Q8. If Other connection type, specify'  ->  'Q20. If Other connection type, specify'
    'Q9. If Other property set-up, specify'  ->  'Q19c. If Other property set-up, speci

## 5. Build the clean database — Step 2: Match to the company database

For every row with an account number, we look it up in the company DB (`acc` column, exact match after trimming/uppercasing). Where a match is found:

- If **`Q2. Customer full name`** is blank, it is filled from the DB's `acc_name`.
- If **`Q4. Phone number / Account Holder`** is blank, it is filled from the DB's `mobile_no`.
- Existing (non-blank) survey answers are **never overwritten** — the DB is only used to fill gaps.
- Reference columns (`DB_Customer_Name`, `DB_Mobile_Number`, `DB_Zone`, `DB_Route`, `DB_Account_Status`, `Match_Method`, `Source File`) are (re)written for every row so you always have an audit trail of what the DB says, even where nothing needed filling.

Account numbers that don't match anything in the DB are flagged in the summary below — they're usually typos from data entry, or genuinely new accounts opened after the DB export was taken.


In [6]:
db = db_df.copy()
db["_acc_key"] = db[DB_COL_ACC].apply(normalize_account)
db = db.drop_duplicates(subset="_acc_key", keep="first")
db_lookup = db.set_index("_acc_key")

db_cols_map = {
    DB_COL_NAME: "DB_Customer_Name",
    DB_COL_PHONE: "DB_Mobile_Number",
    DB_COL_ZONE: "DB_Zone",
    DB_COL_ROUTE: "DB_Route",
    DB_COL_STATUS: "DB_Account_Status",
}

work = new_clean.copy()
work = work.drop(columns=[c for c in db_cols_map.values() if c in work.columns] + ["_source"])
work["_acc_key"] = work[COL_ACCOUNT].apply(normalize_account)

joined = work.join(db_lookup[list(db_cols_map.keys())], on="_acc_key")
joined = joined.rename(columns=db_cols_map)

has_account = joined["_acc_key"].notna()
matched_mask = has_account & joined["DB_Customer_Name"].notna()

name_blank = is_blank(joined[COL_CUST_NAME])
phone_blank = is_blank(joined[COL_CUST_PHONE])
fill_name_mask = name_blank & matched_mask
fill_phone_mask = phone_blank & matched_mask

joined.loc[fill_name_mask, COL_CUST_NAME] = joined.loc[fill_name_mask, "DB_Customer_Name"]
joined.loc[fill_phone_mask, COL_CUST_PHONE] = joined.loc[fill_phone_mask, "DB_Mobile_Number"]

joined["Matched_Account_Number"] = joined[COL_ACCOUNT].where(matched_mask, other=None)
joined["Match_Method"] = pd.Series(np.where(matched_mask, "account_number_exact", ""), index=joined.index).replace("", None)
joined["Source File"] = DB_FILE

not_in_db = has_account & ~matched_mask
final_clean = joined.drop(columns="_acc_key")

print("STEP 2 SUMMARY")
print(f"  Rows with an account number:              {has_account.sum():>6}")
print(f"  Matched to company DB:                     {matched_mask.sum():>6}")
print(f"  Account numbers NOT found in DB (review):  {not_in_db.sum():>6}")
print(f"  Blank customer names filled from DB:        {fill_name_mask.sum():>6}")
print(f"  Blank customer phone numbers filled from DB:{fill_phone_mask.sum():>6}")

if not_in_db.sum():
    print("\n  Sample account numbers not found in DB (for manual review):")
    print("  ", list(final_clean.loc[not_in_db, COL_ACCOUNT].head(10)))


STEP 2 SUMMARY
  Rows with an account number:               13527
  Matched to company DB:                      13346
  Account numbers NOT found in DB (review):     181
  Blank customer names filled from DB:           105
  Blank customer phone numbers filled from DB:   178

  Sample account numbers not found in DB (for manual review):
   ['A02268-001', 'A0394A-001', 'B00138-001', 'B00352-001', 'B00732-001', 'B01738-001', 'B0197A-001', 'B1293-001', 'B1374-001', 'B9502-001']


## 6. Build the clean database — write the workbook

Writes `All Data (Clean)` and `Rejects` back into a single workbook, with a bold header row and frozen top row for readability. Submission-time and interview-date columns are parsed into real dates and displayed as `dd/mm/yyyy`.


In [7]:
clean_db_path = f"{OUTPUT_DIR}/GAWASCO_Clean_Combined_{RUN_DATE_LABEL.replace(' ', '_')}.xlsx"

date_cols_to_format = [DATE_COL_PRIMARY, DATE_COL_FALLBACK]
final_clean_for_write = coerce_dates(final_clean, date_cols_to_format)
new_rejects_sheet_for_write = coerce_dates(new_rejects_sheet, date_cols_to_format)

with pd.ExcelWriter(clean_db_path, engine="openpyxl") as writer:
    final_clean_for_write.to_excel(writer, sheet_name=CLEAN_SHEET, index=False)
    new_rejects_sheet_for_write.to_excel(writer, sheet_name=REJECTS_SHEET, index=False)

wb = openpyxl.load_workbook(clean_db_path)
for sheetname in [CLEAN_SHEET, REJECTS_SHEET]:
    ws = wb[sheetname]
    headers = [c.value for c in ws[1]]
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    ws.freeze_panes = "A2"
    apply_date_number_format(ws, headers, date_cols_to_format)
wb.save(clean_db_path)

print("Saved:", clean_db_path)


Saved: outputs/GAWASCO_Clean_Combined_21st_September.xlsx


## 7. Build the clean database — GPS correction

Moves records whose GPS point landed on the road (not the customer's premise) from `All Data (Clean)` into `Rejects`, using a separately flagged "Wrong Coordinates" workbook (matched by rounded lat/lon). This is the last step in producing the *final* clean database — everything from here on reads the corrected workbook back from disk, so it reflects this correction.

Set `INCLUDE_GPS_CORRECTION_STAGE = False` in the Configuration cell to skip this for a run with no wrong-GPS file; the pipeline still proceeds using `clean_db_path` as-is.


In [8]:
def step_move_wrong_gps(main_path, wrong_gps_path, out_path):
    wb1 = openpyxl.load_workbook(main_path)
    wsc = wb1[CLEAN_SHEET]
    wsr = wb1[REJECTS_SHEET]

    hc = [c.value for c in wsc[1]]
    hr = [c.value for c in wsr[1]]

    lat_idx_c = hc.index(COL_GPS_LAT)
    lon_idx_c = hc.index(COL_GPS_LON)
    lat_idx_r = hr.index(COL_GPS_LAT)
    lon_idx_r = hr.index(COL_GPS_LON)
    acct_idx_c = hc.index(COL_ACCOUNT)
    enum_idx_c = hc.index(COL_COLLECTOR)
    rr_idx = hr.index(COL_REJECT_REASON)

    # Build a map of GPS coordinate -> list of row numbers in the Clean sheet
    clean_rows = []
    for row_num in range(2, wsc.max_row + 1):
        vals = [wsc.cell(row=row_num, column=c + 1).value for c in range(len(hc))]
        clean_rows.append((row_num, vals))

    clean_map = {}
    for row_num, vals in clean_rows:
        key = (round_coord(vals[lat_idx_c]), round_coord(vals[lon_idx_c]))
        clean_map.setdefault(key, []).append(row_num)

    # Existing reject coordinates, so we don't double-process rows already moved
    reject_keys = set()
    for row_num in range(2, wsr.max_row + 1):
        lat = wsr.cell(row=row_num, column=lat_idx_r + 1).value
        lon = wsr.cell(row=row_num, column=lon_idx_r + 1).value
        reject_keys.add((round_coord(lat), round_coord(lon)))

    # Load the flagged wrong-GPS records
    wb2 = openpyxl.load_workbook(wrong_gps_path)
    ws2 = wb2["Wrong Coordinates"]
    h2 = [c.value for c in ws2[1]]
    lat_idx_2 = h2.index("Latitude")
    lon_idx_2 = h2.index("Longitude")

    rows2 = list(ws2.iter_rows(min_row=2, values_only=True))

    to_move, already_done, not_found = [], [], []
    seen_row_nums = set()
    for row2 in rows2:
        lat, lon = row2[lat_idx_2], row2[lon_idx_2]
        key = (round_coord(lat), round_coord(lon))
        if key in clean_map:
            for rn in clean_map[key]:
                if rn not in seen_row_nums:
                    to_move.append(rn)
                    seen_row_nums.add(rn)
        elif key in reject_keys:
            already_done.append(row2)
        else:
            not_found.append(row2)

    print(f"To move from Clean -> Rejects: {len(to_move)}")
    print(f"Already in Rejects (skipped): {len(already_done)}")
    print(f"Not found anywhere (flagged): {len(not_found)}")
    for x in not_found:
        print("  NOT FOUND:", x)

    # Map Reject-sheet columns to Clean-sheet columns by header name
    map_r_to_c = {i: hc.index(name) for i, name in enumerate(hr) if name != COL_REJECT_REASON}

    next_reject_row = wsr.max_row + 1
    for rn in sorted(to_move):
        vals = [wsc.cell(row=rn, column=c + 1).value for c in range(len(hc))]
        acct = vals[acct_idx_c]
        enum_name = vals[enum_idx_c]
        if acct:
            reason = f"Wrong Coordinates - GPS captured on road, not premise (Acct {acct})"
        else:
            reason = f"Wrong Coordinates - GPS captured on road, not premise (matched by coordinates, enum: {enum_name})"
        for r_idx, c_idx in map_r_to_c.items():
            wsr.cell(row=next_reject_row, column=r_idx + 1, value=vals[c_idx])
        wsr.cell(row=next_reject_row, column=rr_idx + 1, value=reason)
        next_reject_row += 1

    # Remove the moved rows from Clean (descending order so row numbers stay valid)
    for rn in sorted(to_move, reverse=True):
        wsc.delete_rows(rn, 1)

    wb1.save(out_path)
    print(f"Saved -> {out_path}")
    print(f"New Clean row count: {wsc.max_row - 1}")
    print(f"New Reject row count: {wsr.max_row - 1}")


if INCLUDE_GPS_CORRECTION_STAGE:
    final_db_path = GPS_STAGE_OUTPUT or clean_db_path
    step_move_wrong_gps(clean_db_path, WRONG_GPS_WORKBOOK, final_db_path)

    # rows moved into Rejects were copied cell-by-cell via raw values, which drops the
    # dd/mm/yyyy display format on the new rows - reapply it to both sheets so it holds
    wb_fmt = openpyxl.load_workbook(final_db_path)
    for sheetname in [CLEAN_SHEET, REJECTS_SHEET]:
        ws_fmt = wb_fmt[sheetname]
        headers_fmt = [c.value for c in ws_fmt[1]]
        apply_date_number_format(ws_fmt, headers_fmt, date_cols_to_format)
    wb_fmt.save(final_db_path)
else:
    final_db_path = clean_db_path
    print("INCLUDE_GPS_CORRECTION_STAGE is False — skipping GPS correction.")

print("Final clean database for this run:", final_db_path)


INCLUDE_GPS_CORRECTION_STAGE is False — skipping GPS correction.
Final clean database for this run: outputs/GAWASCO_Clean_Combined_21st_September.xlsx


## 8. Reload the final clean database

Everything from here on (the progress report, the not-yet-visited PDF, the weekly report, and the day-specific extras) reads `final_db_path` back from disk, so it reflects the GPS correction above (if it ran).


In [9]:
final_clean = pd.read_excel(final_db_path, sheet_name=CLEAN_SHEET)
new_rejects_sheet = pd.read_excel(final_db_path, sheet_name=REJECTS_SHEET)

print(f"Final clean database: {len(final_clean):,} clean rows, {len(new_rejects_sheet):,} reject rows")


Final clean database: 14,681 clean rows, 5,186 reject rows


## 9. Coverage & zone progress

Computed once here and reused everywhere downstream — the Zone Progress sheet, the progress report's zone table, and the not-yet-visited PDF all read from `zone_progress` / `not_visited` / `total_db` / `total_visited` / `pct_covered` below, so they can't disagree with each other.

"Visited" = the account number appears anywhere in the *final* clean sheet (post GPS-correction).


In [10]:
visited_keys = set(final_clean[COL_ACCOUNT].apply(normalize_account).dropna())
db2 = db_df.copy()
db2["_acc_key"] = db2[DB_COL_ACC].apply(normalize_account)
not_visited = db2[~db2["_acc_key"].isin(visited_keys)].copy()
not_visited = not_visited.sort_values([DB_COL_ZONE, DB_COL_ROUTE, DB_COL_ACC])

total_db = len(db2)
total_visited = total_db - len(not_visited)
pct_covered = 100 * total_visited / total_db if total_db else 0

zone_rows = []
for zone, zdf in db2.groupby(DB_COL_ZONE, sort=True):
    visited_in_zone = int(zdf["_acc_key"].isin(visited_keys).sum())
    total_in_zone = len(zdf)
    not_visited_in_zone = total_in_zone - visited_in_zone
    cov = 100 * visited_in_zone / total_in_zone if total_in_zone else 0
    zone_rows.append({
        "Zone": zone,
        "Total Accounts": total_in_zone,
        "Visited": visited_in_zone,
        "Not Visited": not_visited_in_zone,
        "Coverage %": round(cov, 1),
    })
zone_progress = pd.DataFrame(zone_rows).sort_values("Zone").reset_index(drop=True)

print(f"Total DB accounts: {total_db} | Visited: {total_visited} | Not yet visited: {len(not_visited)} | Coverage: {pct_covered:.1f}%")
zone_progress


Total DB accounts: 22162 | Visited: 13346 | Not yet visited: 8816 | Coverage: 60.2%


,Zone,Total Accounts,Visited,Not Visited,Coverage %
0,A,639,558,81,87.3
1,B,925,898,27,97.1
2,C,722,662,60,91.7
3,D,668,668,0,100.0
4,E,1910,1877,33,98.3
5,F,1984,1404,580,70.8
6,G,673,593,80,88.1
7,H,1367,949,418,69.4
8,J,1114,1050,64,94.3
9,K,1681,1645,36,97.9


## 10. Zone Progress sheet (in the clean database workbook)

Adds a **`Zone Progress`** sheet onto `final_db_path` (alongside `All Data (Clean)` and `Rejects`) — a live, at-a-glance tracker of survey coverage by zone: total accounts, visited, not visited, and coverage %, with a colour scale on the coverage column, a stacked bar chart (Visited vs Not Visited per zone), and a grand-total row. Every run overwrites this sheet with the current numbers, so it always reflects the latest state of the database.


In [11]:
def write_zone_progress_sheet(workbook_path, zone_progress_df, sheet_name="Zone Progress"):
    wb = openpyxl.load_workbook(workbook_path)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)

    headers = ["Zone", "Total Accounts", "Visited", "Not Visited", "Coverage %"]
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Arial", bold=True, color="FFFFFF")
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    ws.cell(row=1, column=1, value=f"Zone Progress — as of {datetime.now().strftime('%d %B %Y %H:%M')}")
    ws.cell(row=1, column=1).font = Font(name="Arial", bold=True, italic=True, size=11)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(headers))

    header_row = 2
    for col, h in enumerate(headers, start=1):
        cell = ws.cell(row=header_row, column=col, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")
        cell.border = border

    start_row = header_row + 1
    for i, row in enumerate(zone_progress_df.itertuples(index=False, name=None)):
        r_ = start_row + i
        zone_val, total_val, visited_val, not_visited_val, coverage_val = row
        ws.cell(row=r_, column=1, value=zone_val).font = Font(name="Arial")
        ws.cell(row=r_, column=2, value=int(total_val)).font = Font(name="Arial")
        ws.cell(row=r_, column=3, value=int(visited_val)).font = Font(name="Arial")
        ws.cell(row=r_, column=4, value=int(not_visited_val)).font = Font(name="Arial")
        ws.cell(row=r_, column=5, value=float(coverage_val)).font = Font(name="Arial")
        ws.cell(row=r_, column=5).number_format = "0.0"
        for c in range(1, 6):
            ws.cell(row=r_, column=c).border = border
            if c in (2, 3, 4, 5):
                ws.cell(row=r_, column=c).alignment = Alignment(horizontal="center")

    end_row = start_row + len(zone_progress_df) - 1
    total_row = end_row + 1
    total_accounts_sum = int(zone_progress_df["Total Accounts"].sum())
    visited_sum = int(zone_progress_df["Visited"].sum())
    not_visited_sum = int(zone_progress_df["Not Visited"].sum())
    total_coverage = round(100 * visited_sum / total_accounts_sum, 1) if total_accounts_sum else 0.0

    ws.cell(row=total_row, column=1, value="TOTAL").font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=2, value=total_accounts_sum).font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=3, value=visited_sum).font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=4, value=not_visited_sum).font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=5, value=total_coverage).font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=5).number_format = "0.0"
    for c in range(1, 6):
        cell = ws.cell(row=total_row, column=c)
        cell.border = border
        cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
        if c in (2, 3, 4, 5):
            cell.alignment = Alignment(horizontal="center")

    # colour-scale the coverage column (red -> yellow -> green) so low-coverage zones jump out
    coverage_range = f"E{start_row}:E{end_row}"
    if end_row >= start_row:
        rule = ColorScaleRule(
            start_type="min", start_color="F8696B",
            mid_type="percentile", mid_value=50, mid_color="FFEB84",
            end_type="max", end_color="63BE7B",
        )
        ws.conditional_formatting.add(coverage_range, rule)

    # stacked bar chart: Visited vs Not Visited, per zone
    if end_row >= start_row:
        chart = BarChart()
        chart.type = "col"
        chart.grouping = "stacked"
        chart.overlap = 100
        chart.title = "Zone Coverage — Visited vs Not Visited"
        chart.y_axis.title = "Accounts"
        chart.x_axis.title = "Zone"
        chart.width = 18
        chart.height = 10

        data = Reference(ws, min_col=3, max_col=4, min_row=header_row, max_row=end_row)
        cats = Reference(ws, min_col=1, min_row=start_row, max_row=end_row)
        chart.add_data(data, titles_from_data=True)
        chart.set_categories(cats)
        ws.add_chart(chart, f"G{header_row}")

    ws.column_dimensions["A"].width = 22
    for col in "BCDE":
        ws.column_dimensions[col].width = 15
    ws.freeze_panes = "A3"

    wb.save(workbook_path)
    print(f"Zone Progress sheet saved: {len(zone_progress_df)} zones, overall coverage {total_coverage:.1f}%")


write_zone_progress_sheet(final_db_path, zone_progress)


Zone Progress sheet saved: 13 zones, overall coverage 60.2%


## 11. Daily enumerator report

Accepted / Rejected / Total per data collector, for `REPORT_DATE`, read from the final (corrected) database via `build_daily_counts`. Feeds into the progress report below, and is recomputed identically for the Enumerator Summary sheet later on.


In [12]:
accepted_today, rejected_today = build_daily_counts(final_db_path, REPORT_DATE)
daily_report_with_total = daily_counts_to_frame(accepted_today, rejected_today)

print(f"Daily enumerator report for {REPORT_DATE}:")
daily_report_with_total


Daily enumerator report for 2026-09-21:


,Accepted,Rejected,Total
Abdullahi Hassan Keynan,24,1,25
Abdullahi Uweso Dara,20,2,22
Ahmed Abdi Aden,0,11,11
Fathi Hassan Issa,6,0,6
Ismail Ali Abdullahi,5,1,6
Mohamud Mohamed Abdi,21,0,21
Najma Aden Shide,15,1,16
TOTAL,91,16,107


## 12. Progress report

A management-facing summary: overall coverage, this run's processing stats, zone-wise coverage (table + a colour-coded coverage-by-zone bar chart), the daily enumerator performance table (for `REPORT_DATE`), and data-quality notes — built from the final clean database (reusing `zone_progress` from above).


In [13]:
out3_path = f"{OUTPUT_DIR}/GAWASCO_Progress_Report_{RUN_DATE_LABEL.replace(' ', '_')}.docx"

d = Document()
d.add_heading("GAWASCO Customer Identification Survey — Progress Report", level=0)
p = d.add_paragraph(f"Generated: {datetime.now().strftime('%d %B %Y, %H:%M')}")
p.runs[0].italic = True

d.add_heading("1. Overall Progress", level=1)
table = d.add_table(rows=1, cols=2)
table.style = "Light Grid Accent 1"
table.rows[0].cells[0].text, table.rows[0].cells[1].text = "Metric", "Value"

not_found_in_db = (final_clean[COL_ACCOUNT].apply(normalize_account).notna() & final_clean["DB_Customer_Name"].isna()).sum()
new_dup_count = (dup_rows["_source"] == "new").sum()

daily_total_accepted = int(daily_report_with_total.loc["TOTAL", "Accepted"]) if len(daily_report_with_total) else 0
daily_total_rejected = int(daily_report_with_total.loc["TOTAL", "Rejected"]) if len(daily_report_with_total) else 0

rows_data = [
    ("Total accounts in company database", f"{total_db:,}"),
    ("Accounts visited / surveyed to date", f"{total_visited:,}"),
    ("Accounts not yet visited", f"{len(not_visited):,}"),
    ("Overall coverage", f"{pct_covered:.1f}%"),
    ("New submissions processed this run", f"{len(raw_df):,}"),
    (f"  of which accepted on {REPORT_DATE_LABEL} (Account Holder / Potential Customer)", f"{daily_total_accepted:,}"),
    (f"  of which rejected on {REPORT_DATE_LABEL} (no account number / duplicate / GPS)", f"{daily_total_rejected:,}"),
    ("Duplicate submissions found this run", f"{new_dup_count:,}"),
    ("Potential customers identified (no existing account)", f"{(final_clean[COL_RECORD_TYPE] == 'Potential Customer').sum():,}"),
    ("Customer names filled in from company DB this run", f"{fill_name_mask.sum():,}"),
    ("Customer phone numbers filled in from company DB this run", f"{fill_phone_mask.sum():,}"),
    ("Account numbers not found in company DB (review for typos)", f"{not_found_in_db:,}"),
]
for k, v in rows_data:
    row = table.add_row().cells
    row[0].text, row[1].text = k, v

d.add_heading("2. Zone-wise Coverage", level=1)
zone_tbl = d.add_table(rows=1, cols=4)
zone_tbl.style = "Light Grid Accent 1"
h = zone_tbl.rows[0].cells
h[0].text, h[1].text, h[2].text, h[3].text = "Zone", "Total Accounts", "Visited", "Coverage %"
for _, zrow in zone_progress.iterrows():
    r = zone_tbl.add_row().cells
    r[0].text, r[1].text, r[2].text, r[3].text = str(zrow["Zone"]), str(int(zrow["Total Accounts"])), str(int(zrow["Visited"])), f"{zrow['Coverage %']:.1f}%"

# coverage-by-zone bar chart, colour-coded red/amber/green so low-coverage zones stand out
zone_chart_path = f"{OUTPUT_DIR}/_zone_coverage_chart_{RUN_DATE_LABEL.replace(' ', '_')}.png"
zp_sorted = zone_progress.sort_values("Coverage %")
bar_colors = ["#C00000" if c < 50 else "#ED9C28" if c < 80 else "#2E9E4C" for c in zp_sorted["Coverage %"]]

fig, ax = plt.subplots(figsize=(7, max(2.2, 0.4 * len(zp_sorted))))
ax.barh(zp_sorted["Zone"].astype(str), zp_sorted["Coverage %"], color=bar_colors)
for i, (zone, cov) in enumerate(zip(zp_sorted["Zone"].astype(str), zp_sorted["Coverage %"])):
    ax.text(cov + 1, i, f"{cov:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 105)
ax.set_xlabel("Coverage %")
ax.set_title("Coverage by Zone")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(zone_chart_path, dpi=150)
plt.close(fig)

d.add_picture(zone_chart_path, width=Inches(6))
os.remove(zone_chart_path)  # embedded in the doc now, don't leave a stray PNG in the outputs folder

d.add_heading("3. Daily Enumerator Performance", level=1)
d.add_paragraph(f"Submissions on {REPORT_DATE_LABEL} — accepted vs rejected, by data collector (post GPS-correction).")
enum_tbl = d.add_table(rows=1, cols=4)
enum_tbl.style = "Light Grid Accent 1"
eh = enum_tbl.rows[0].cells
eh[0].text, eh[1].text, eh[2].text, eh[3].text = "Data Collector", "Accepted", "Rejected", "Total"
for name, row in daily_report_with_total.iterrows():
    r = enum_tbl.add_row().cells
    r[0].text = str(name)
    r[1].text = str(int(row["Accepted"]))
    r[2].text = str(int(row["Rejected"]))
    r[3].text = str(int(row["Total"]))

d.add_heading("4. Notes", level=1)
d.add_paragraph(
    "Accounts with a survey account number that could not be matched against the company "
    "database should be reviewed for possible typos, or may represent accounts opened after "
    "the last database export."
)
d.add_paragraph(
    "Rejected submissions (no account number, with service marked available or the question left "
    "blank; duplicates; or GPS captured on the road rather than the premise) are kept in the Rejects "
    "sheet with a reason, for follow-up with the relevant enumerator."
)
d.add_paragraph(
    "The full list of not-yet-visited accounts, organized zone by zone and route by route, "
    "is provided in the accompanying PDF report."
)

d.save(out3_path)
print("Saved:", out3_path)


Saved: outputs/GAWASCO_Progress_Report_21st_September.docx


## 13. Remaining accounts (PDF)

"Visited" = the account number appears anywhere in the *final* clean sheet (post GPS-correction). Every DB account not in that set is listed, grouped by zone and then by route, each with its own table.


In [14]:
out2_path = f"{OUTPUT_DIR}/GAWASCO_Not_Visited_Accounts_{RUN_DATE_LABEL.replace(' ', '_')}.pdf"

styles = getSampleStyleSheet()
zone_style = ParagraphStyle("zone", parent=styles["Heading1"], fontSize=14, spaceBefore=14, spaceAfter=6, textColor=colors.HexColor("#1F4E78"))
route_style = ParagraphStyle("route", parent=styles["Heading2"], fontSize=11, spaceBefore=8, spaceAfter=4, textColor=colors.HexColor("#2E75B6"))

doc = SimpleDocTemplate(out2_path, pagesize=A4, topMargin=1.5*cm, bottomMargin=1.5*cm, leftMargin=1.5*cm, rightMargin=1.5*cm)
story = [
    Paragraph("GAWASCO Survey — Accounts Not Yet Visited", styles["Title"]),
    Paragraph(f"Generated: {datetime.now().strftime('%d %B %Y %H:%M')}", styles["Normal"]),
    Paragraph(f"Total accounts in company database: {total_db} | Visited: {total_visited} | Not yet visited: {len(not_visited)}", styles["Normal"]),
    Spacer(1, 12),
]

for zone, zdf in not_visited.groupby(DB_COL_ZONE, sort=True):
    story.append(Paragraph(f"Zone {zone}  ({len(zdf)} not visited)", zone_style))
    for route, rdf in zdf.groupby(DB_COL_ROUTE, sort=True):
        story.append(Paragraph(f"Route {route}  ({len(rdf)} accounts)", route_style))
        data = [["Account No", "Customer Name", "Mobile No", "Category", "Status"]]
        for _, r in rdf.iterrows():
            data.append([str(r[DB_COL_ACC]), str(r[DB_COL_NAME])[:35], str(r[DB_COL_PHONE]), str(r[DB_COL_CATEGORY]), str(r[DB_COL_STATUS])])
        t = Table(data, colWidths=[2.6*cm, 6.2*cm, 3.2*cm, 3*cm, 2.5*cm], repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F4E78")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTSIZE", (0, 0), (-1, -1), 7.5),
            ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F2F2")]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ]))
        story.append(t)
        story.append(Spacer(1, 6))
    story.append(PageBreak())

doc.build(story)
print("Saved:", out2_path)


Saved: outputs/GAWASCO_Not_Visited_Accounts_21st_September.pdf


## 14. Weekly Submissions Report

A single Excel file covering **every submission (accepted + rejected) in the selected date range**, sourced from the final (post GPS-correction) `final_clean` / `new_rejects_sheet` above. Weeks run Sunday–Saturday by default; set `DATE_RANGE_MODE = "custom"` in the Configuration cell for an arbitrary range. Submission time is used as the primary date (interview date as fallback) for placing a row in range, same as everywhere else.

Two sheets:
- **`Weekly Summary`** — each data collector's Accepted/Rejected counts for every day in range, plus Total Accepted / Total Rejected / Grand Total and a grand-total row.
- **`Submissions`** — the individual records (accepted and rejected) in range, with an `Outcome` column; submission-time and interview-date columns shown as `dd/mm/yyyy`.


In [15]:
clean_for_week = final_clean.copy()
rejects_for_week = new_rejects_sheet.copy()
clean_for_week["Outcome"] = "Accepted"
rejects_for_week["Outcome"] = "Rejected"

all_cols = list(dict.fromkeys(list(clean_for_week.columns) + list(rejects_for_week.columns)))
week_df = pd.concat([clean_for_week.reindex(columns=all_cols), rejects_for_week.reindex(columns=all_cols)], ignore_index=True)

parsed = week_df[DATE_COL_PRIMARY].apply(parse_any_date)
fallback = week_df[DATE_COL_FALLBACK].apply(parse_any_date)
parsed = parsed.fillna(fallback)
parsed = pd.to_datetime(parsed, errors="coerce")
week_df["_parsed_date"] = parsed.dt.normalize()

if DATE_RANGE_MODE == "week":
    range_start, range_end = sunday_saturday_week(ANY_DATE_IN_WEEK)
elif DATE_RANGE_MODE == "custom":
    range_start = pd.Timestamp(CUSTOM_START_DATE).normalize()
    range_end = pd.Timestamp(CUSTOM_END_DATE).normalize()
else:
    raise ValueError('DATE_RANGE_MODE must be "week" or "custom"')

print(f"Weekly report range: {range_start.date()} ({range_start.day_name()}) to {range_end.date()} ({range_end.day_name()})")

week_mask = (week_df["_parsed_date"] >= range_start) & (week_df["_parsed_date"] <= range_end)
week_filtered = week_df[week_mask].copy().sort_values(["_parsed_date", COL_COLLECTOR])
print(f"Submissions in range: {len(week_filtered):,} (Accepted: {(week_filtered['Outcome']=='Accepted').sum():,}, Rejected: {(week_filtered['Outcome']=='Rejected').sum():,})")

all_days = pd.date_range(range_start, range_end, freq="D")
day_labels = [dd.strftime("%a %d-%b") for dd in all_days]
day_map = dict(zip(all_days, day_labels))
week_filtered["_day_label"] = week_filtered["_parsed_date"].map(day_map)

week_pivot = week_filtered.groupby([COL_COLLECTOR, "_day_label", "Outcome"]).size().unstack(["_day_label", "Outcome"], fill_value=0)
col_order = pd.MultiIndex.from_product([day_labels, ["Accepted", "Rejected"]])
week_pivot = week_pivot.reindex(columns=col_order, fill_value=0)
week_pivot.columns = [f"{dl} ({o})" for dl, o in week_pivot.columns]

tot_acc = week_filtered[week_filtered["Outcome"] == "Accepted"].groupby(COL_COLLECTOR).size()
tot_rej = week_filtered[week_filtered["Outcome"] == "Rejected"].groupby(COL_COLLECTOR).size()
week_pivot["Total Accepted"] = tot_acc.reindex(week_pivot.index, fill_value=0)
week_pivot["Total Rejected"] = tot_rej.reindex(week_pivot.index, fill_value=0)
week_pivot["Grand Total"] = week_pivot["Total Accepted"] + week_pivot["Total Rejected"]
week_pivot = week_pivot.sort_index()

week_grand = week_pivot.sum(numeric_only=True)
week_grand.name = "TOTAL"
week_summary_with_total = pd.concat([week_pivot, week_grand.to_frame().T])

out4_path = f"{OUTPUT_DIR}/GAWASCO_Weekly_Submissions_Report_{RUN_DATE_LABEL.replace(' ', '_')}.xlsx"

week_submissions_out = week_filtered.drop(columns=["_parsed_date", "_day_label"])
week_summary_out = week_summary_with_total.reset_index().rename(columns={"index": COL_COLLECTOR})

with pd.ExcelWriter(out4_path, engine="openpyxl") as writer:
    week_summary_out.to_excel(writer, sheet_name="Weekly Summary", index=False)
    week_submissions_out.to_excel(writer, sheet_name="Submissions", index=False)

wb4 = openpyxl.load_workbook(out4_path)
ws4 = wb4["Weekly Summary"]
for cell in ws4[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    cell.alignment = Alignment(horizontal="center")
for cell in ws4[ws4.max_row]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
ws4.freeze_panes = "B2"
for col_idx in range(1, ws4.max_column + 1):
    ws4.column_dimensions[get_column_letter(col_idx)].width = 16

ws5 = wb4["Submissions"]
for cell in ws5[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
ws5.freeze_panes = "A2"
apply_date_number_format(ws5, [c.value for c in ws5[1]], [DATE_COL_PRIMARY, DATE_COL_FALLBACK])

wb4.save(out4_path)
print("Saved:", out4_path)


Weekly report range: 2026-09-20 (Sunday) to 2026-09-26 (Saturday)
Submissions in range: 128 (Accepted: 106, Rejected: 22)
Saved: outputs/GAWASCO_Weekly_Submissions_Report_21st_September.xlsx


## 15. Day-specific extras

Three more things, added onto `final_db_path` (the same corrected workbook), all filtered to `REPORT_DATE`:

1. **Enumerator Submission Summary sheet** — Accepted / Rejected / Total per data collector (same numbers as the progress report's daily table, via `build_daily_counts`).
2. **Reject List sheet** — the reject rows for `REPORT_DATE`, with respondent contact details and reject reason.
3. **Word analysis report** — a narrative write-up (numbers, tables, and commentary) for `REPORT_DATE`.


### 15.1 Enumerator Submission Summary sheet

In [16]:
def step_enumerator_summary(workbook_path, report_date, sheet_name):
    accepted, rejected = build_daily_counts(workbook_path, report_date)
    enums = sorted(set(accepted) | set(rejected))

    wb = openpyxl.load_workbook(workbook_path)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)

    headers = ["Enumerator", "Accepted", "Rejected", "Total"]
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Arial", bold=True, color="FFFFFF")
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for col, h in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")
        cell.border = border

    start_row = 2
    for i, e in enumerate(enums):
        r_ = start_row + i
        ws.cell(row=r_, column=1, value=e).font = Font(name="Arial")
        ws.cell(row=r_, column=2, value=accepted[e]).font = Font(name="Arial")
        ws.cell(row=r_, column=3, value=rejected[e]).font = Font(name="Arial")
        ws.cell(row=r_, column=4, value=f"=B{r_}+C{r_}").font = Font(name="Arial")
        for c in range(1, 5):
            ws.cell(row=r_, column=c).border = border
            if c in (2, 3, 4):
                ws.cell(row=r_, column=c).alignment = Alignment(horizontal="center")

    end_row = start_row + len(enums) - 1
    total_row = end_row + 1
    ws.cell(row=total_row, column=1, value="TOTAL").font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=2, value=f"=SUM(B{start_row}:B{end_row})").font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=3, value=f"=SUM(C{start_row}:C{end_row})").font = Font(name="Arial", bold=True)
    ws.cell(row=total_row, column=4, value=f"=SUM(D{start_row}:D{end_row})").font = Font(name="Arial", bold=True)
    for c in range(1, 5):
        cell = ws.cell(row=total_row, column=c)
        cell.border = border
        cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
        if c in (2, 3, 4):
            cell.alignment = Alignment(horizontal="center")

    ws.column_dimensions["A"].width = 30
    for col in "BCD":
        ws.column_dimensions[col].width = 12
    ws.freeze_panes = "A2"

    wb.save(workbook_path)
    print(f"Enumerators: {len(enums)} | Total row: {total_row}")


step_enumerator_summary(final_db_path, REPORT_DATE, sheet_name=f"Enumerator Summary ({REPORT_DATE_LABEL})")


Enumerators: 7 | Total row: 9


### 15.2 Reject List sheet

In [17]:
def step_reject_list(workbook_path, report_date, sheet_name):
    wb = openpyxl.load_workbook(workbook_path)
    wsr = wb[REJECTS_SHEET]
    hr = [c.value for c in wsr[1]]

    cols_needed = [
        COL_COLLECTOR,
        COL_ZONE,
        COL_ROUTE,
        COL_RESPONDENT_PHONE,
        COL_RESPONDENT_NAME,
        COL_ACCOUNT,
        COL_REJECT_REASON,
    ]
    pi_r = hr.index(DATE_COL_PRIMARY)
    fi_r = hr.index(DATE_COL_FALLBACK)
    idxs = [hr.index(c) for c in cols_needed]

    rows_out = []
    for row in wsr.iter_rows(min_row=2, values_only=True):
        if matches_report_date(row[pi_r], row[fi_r], report_date):
            rows_out.append([row[i] for i in idxs])

    print(f"Matching reject rows: {len(rows_out)}")

    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)

    headers = ["Data Collector Name", "Zone", "Route", "Respondent's Phone Number",
               "Respondent's Name", "Account Number", "Reason for Reject"]

    header_fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    header_font = Font(name="Arial", bold=True, color="FFFFFF")
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for col, h in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
        cell.border = border

    for i, rowvals in enumerate(rows_out, start=2):
        for c, v in enumerate(rowvals, start=1):
            cell = ws.cell(row=i, column=c, value=v)
            cell.font = Font(name="Arial")
            cell.border = border
            if c in (2, 3, 4, 6):
                cell.alignment = Alignment(horizontal="center")

    widths = [26, 10, 14, 20, 26, 16, 60]
    for c, w in enumerate(widths, start=1):
        ws.column_dimensions[chr(64 + c)].width = w
    ws.freeze_panes = "A2"

    wb.save(workbook_path)
    print("Reject list sheet saved.")


step_reject_list(final_db_path, REPORT_DATE, sheet_name=f"Reject List ({REPORT_DATE_LABEL})")


Matching reject rows: 16
Reject list sheet saved.


### 15.3 Word analysis report

> **Note carried over from the original script:** the narrative paragraphs below were written for a specific day's findings (e.g. duplicate accounts as the #1 cause, particular zone callouts). If the numbers tell a different story for `REPORT_DATE` this run, rewrite the narrative paragraphs to match what the data actually shows — don't just trust the old wording.


In [ ]:
def step_word_analysis_report(workbook_path, report_date, report_date_label_full, report_out):
    NAVY = RGBColor(0x1F, 0x4E, 0x78)
    GREY = RGBColor(0x59, 0x59, 0x59)

    def set_cell_shading(cell, hex_color):
        shd = OxmlElement("w:shd")
        shd.set(qn("w:val"), "clear")
        shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), hex_color)
        cell._tc.get_or_add_tcPr().append(shd)

    def style_header_row(row, hex_color="1F4E78"):
        for cell in row.cells:
            set_cell_shading(cell, hex_color)
            for p in cell.paragraphs:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                for run in p.runs:
                    run.font.bold = True
                    run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER

    # ---- Pull the numbers straight from the workbook ----
    wb = openpyxl.load_workbook(workbook_path)
    wsc = wb[CLEAN_SHEET]
    wsr = wb[REJECTS_SHEET]
    hc = [c.value for c in wsc[1]]
    hr = [c.value for c in wsr[1]]

    pi_c, zone_c, en_c = hc.index(DATE_COL_PRIMARY), hc.index(COL_ZONE), hc.index(COL_COLLECTOR)
    fi_c = hc.index(DATE_COL_FALLBACK)
    pi_r, zone_r, en_r = hr.index(DATE_COL_PRIMARY), hr.index(COL_ZONE), hr.index(COL_COLLECTOR)
    fi_r = hr.index(DATE_COL_FALLBACK)
    reason_r = hr.index(COL_REJECT_REASON)

    acc_zone, rej_zone, reasons = collections.Counter(), collections.Counter(), collections.Counter()
    acc_by_enum, rej_by_enum = collections.Counter(), collections.Counter()
    rej_reason_by_enum = collections.defaultdict(collections.Counter)
    acc_total = rej_total = 0

    for row in wsc.iter_rows(min_row=2, values_only=True):
        if matches_report_date(row[pi_c], row[fi_c], report_date):
            acc_total += 1
            acc_zone[row[zone_c]] += 1
            acc_by_enum[row[en_c]] += 1

    for row in wsr.iter_rows(min_row=2, values_only=True):
        if matches_report_date(row[pi_r], row[fi_r], report_date):
            rej_total += 1
            rej_zone[row[zone_r]] += 1
            rej_by_enum[row[en_r]] += 1
            reason = row[reason_r] or "Unknown"
            if "Wrong Coordinates" in reason:
                reasons["Wrong GPS coordinates (on road, not premise)"] += 1
                rej_reason_by_enum[row[en_r]]["GPS"] += 1
            elif "No account number" in reason:
                reasons["No account number captured"] += 1
                rej_reason_by_enum[row[en_r]]["NoAcct"] += 1
            elif "Duplicate account" in reason:
                reasons["Duplicate account number"] += 1
                rej_reason_by_enum[row[en_r]]["Dup"] += 1
            else:
                reasons[reason] += 1
                rej_reason_by_enum[row[en_r]]["Other"] += 1

    total = acc_total + rej_total
    print(f"Accepted {acc_total}, Rejected {rej_total}, Total {total}")

    # ---- Build the document ----
    doc = Document()
    section = doc.sections[0]
    section.page_width, section.page_height = Cm(21.59), Cm(27.94)  # US Letter
    for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
        setattr(section, m, Cm(1.9))

    style = doc.styles["Normal"]
    style.font.name = "Arial"
    style.font.size = Pt(11)

    title = doc.add_paragraph()
    r = title.add_run("GAWASCO Field Data Collection")
    r.bold, r.font.size, r.font.color.rgb = True, Pt(22), NAVY

    sub = doc.add_paragraph()
    r = sub.add_run(f"Daily Summary & Analysis — {report_date_label_full}")
    r.italic, r.font.size, r.font.color.rgb = True, Pt(13), GREY

    doc.add_paragraph()  # spacer

    doc.add_heading("How the day went", level=1)
    if total:
        doc.add_paragraph(
            f"Enumerators between them logged {total:,} submissions on {report_date_label_full}, and the large "
            f"majority of that work held up well: {acc_total:,} records (about {acc_total/total*100:.0f}%) passed "
            f"quality checks and are now sitting in the clean dataset. The remaining {rej_total:,} records (roughly "
            f"{rej_total/total*100:.0f}%) were rejected, mostly for reasons that are fixable rather than alarming — "
            f"this reads more like normal fieldwork friction than a systemic problem."
        )
    else:
        doc.add_paragraph(
            f"No submissions were recorded for {report_date_label_full} in the Clean or Rejects sheets."
        )
    doc.add_paragraph(
        "The short version: most of the team is submitting clean, usable data, a handful of enumerators and "
        "zones are driving a disproportionate share of the rejections, and the rejection reasons themselves "
        "point to two or three coaching opportunities rather than a systemic problem."
    )

    doc.add_heading("The numbers at a glance", level=1)
    t = doc.add_table(rows=1, cols=2)
    t.style = "Table Grid"
    t.rows[0].cells[0].text, t.rows[0].cells[1].text = "Metric", "Value"
    style_header_row(t.rows[0])
    data = [
        ("Total submissions", f"{total:,}"),
        ("Accepted (clean)", f"{acc_total:,}  ({(acc_total/total*100 if total else 0):.1f}%)"),
        ("Rejected", f"{rej_total:,}  ({(rej_total/total*100 if total else 0):.1f}%)"),
        ("Enumerators active", str(len(set(acc_by_enum) | set(rej_by_enum)))),
    ]
    for label, val in data:
        row = t.add_row()
        row.cells[0].text, row.cells[1].text = label, val
        row.cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_heading("Why records were rejected", level=1)
    doc.add_paragraph(
        "Duplicate account numbers are typically the biggest driver of rejections — the same customer account "
        "being captured more than once. GPS points landing on the road instead of the customer's premise is "
        "another common issue, and a smaller batch of forms often goes out with no account number at all even "
        "though the respondent indicated the service was available."
    )
    if rej_total:
        t = doc.add_table(rows=1, cols=3)
        t.style = "Table Grid"
        t.rows[0].cells[0].text, t.rows[0].cells[1].text, t.rows[0].cells[2].text = "Reason", "Records", "Share"
        style_header_row(t.rows[0])
        for reason, count in reasons.most_common():
            row = t.add_row()
            row.cells[0].text = reason
            row.cells[1].text = str(count)
            row.cells[2].text = f"{count/rej_total*100:.0f}%"
            row.cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
            row.cells[2].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_heading("Zone by zone", level=1)
    zones = sorted(set(acc_zone) | set(rej_zone), key=lambda z: (rej_zone[z]/(acc_zone[z]+rej_zone[z])) if (acc_zone[z]+rej_zone[z]) else 0)
    doc.add_paragraph(
        "Zone-level rejection rates vary widely — some zones are performing very well, while one or two "
        "small-volume zones show elevated rejection rates worth a closer look. See the table below for the "
        "full breakdown; interpret low-volume zones with caution."
    )
    t = doc.add_table(rows=1, cols=5)
    t.style = "Table Grid"
    hdrs = ["Zone", "Accepted", "Rejected", "Total", "Reject rate"]
    for i, h in enumerate(hdrs):
        t.rows[0].cells[i].text = h
    style_header_row(t.rows[0])
    for z in zones:
        a, rj = acc_zone[z], rej_zone[z]
        tt = a + rj
        rate = rj / tt * 100 if tt else 0
        row = t.add_row()
        zone_label = str(z) if z is not None else "(blank)"
        vals = [zone_label, str(a), str(rj), str(tt), f"{rate:.1f}%"]
        for i, v in enumerate(vals):
            row.cells[i].text = v
            if i > 0:
                row.cells[i].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_heading("A note on individual enumerators", level=1)
    doc.add_paragraph(
        "Review the per-enumerator Accepted/Rejected/Total counts (see the Enumerator Summary sheet) to spot "
        "outliers — both very high rejection rates worth a coaching conversation, and very high volume "
        "performers worth recognizing. For each flagged enumerator, check which rejection reason dominates "
        "(duplicate account, missing account number, or GPS) before deciding what feedback to give — the "
        "fix is different for each."
    )

    doc.add_heading("What this suggests", level=1)
    for bullet in [
        "Tackle duplicate account numbers first — it's typically the largest single driver of rejections and "
        "likely reflects either re-visits to the same household or a search step being skipped before starting "
        "a new form.",
        "Remind teams to always capture an account number when a respondent confirms service is available.",
        "Treat low-volume zones/enumerators with caution — a handful of rejections can swing the rate a lot; "
        "watch over several days before acting on one day's data.",
        "Use the best-performing zones as a benchmark, or even a source of peer mentors, for the higher-rejection ones.",
    ]:
        doc.add_paragraph(bullet, style="List Bullet")

    footer = doc.add_paragraph()
    r = footer.add_run(
        f"Source: {CLEAN_SHEET} and {REJECTS_SHEET} sheets, {workbook_path} — filtered to {report_date} (by submission time, falling back to interview date)."
    )
    r.italic, r.font.size, r.font.color.rgb = True, Pt(9), GREY

    doc.save(report_out)
    print(f"Report saved -> {report_out}")


step_word_analysis_report(final_db_path, REPORT_DATE, REPORT_DATE_LABEL_FULL, WORD_ANALYSIS_REPORT_OUT)


## 16. Done

**Clean database (Step 1):**
- `final_db_path` — the final clean/rejects workbook, GPS-corrected if `INCLUDE_GPS_CORRECTION_STAGE = True` (feed this in as next run's `CLEAN_FILE`)
- `final_db_path` also carries a **`Zone Progress`** sheet — total/visited/not-visited/coverage % by zone, colour-scaled, with a Visited-vs-Not-Visited bar chart — refreshed every run

**Progress report (Step 2):**
- `out3_path` — coverage, this run's stats, zone breakdown (table + coverage-by-zone bar chart), daily enumerator table (Word)

**Remaining accounts (Step 3):**
- `out2_path` — not-yet-visited accounts, zone/route wise (PDF)

**Supporting outputs:**
- `out4_path` — weekly submissions report, accepted/rejected by collector (Excel)
- `final_db_path` also carries two more sheets: `Enumerator Summary (...)` and `Reject List (...)`, both filtered to `REPORT_DATE`
- `WORD_ANALYSIS_REPORT_OUT` — the day-specific narrative Word report

**Next time:** set `RAW_FILE` to the new raw export, `CLEAN_FILE` to `final_db_path` from this run, `DB_FILE` to the latest company DB export, `WRONG_GPS_WORKBOOK` and `REPORT_DATE`/`REPORT_DATE_LABEL(_FULL)` to this new run's values, and `ANY_DATE_IN_WEEK` (or the custom dates) to the week you want reported, then re-run.


In [ ]:
print("Clean database (final, post-GPS-correction, incl. Zone Progress sheet):", final_db_path)
print("Progress report:                            ", out3_path)
print("Remaining accounts (not-visited PDF):        ", out2_path)
print("Weekly submissions report:                   ", out4_path)
print("Word analysis report:                        ", WORD_ANALYSIS_REPORT_OUT)
